In [0]:
import yaml

current_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Get the directory containing the notebook (remove the notebook filename)
src_directory = '/'.join(current_notebook_path.split('/')[:-1])

# Build the CSV file path
file_path = f"/Workspace{src_directory}/util/user_group_privilege_parameters.yml"
print(f"CSV file path: {file_path}")

with open(file_path, "r") as file:
    config = yaml.safe_load(file)

In [0]:
user_names = config['USERS']
group_name = config['GROUP_NAME']
table_name = config['TABLE_NAME']
permissions = config['PERMISSIONS']
flag = config['FLAG']

In [0]:
if not user_names:
    raise ValueError("Missing user_names parameter")
if not group_name:
    raise ValueError("Missing group_name parameter")

In [0]:
import requests
import json

In [0]:
scope_name = config['SCOPE_NAME']
account_id = dbutils.secrets.get(scope_name,  config["SECRETS"]["ACCOUNT_ID"] )
DATABRICKS_Account_URL = dbutils.secrets.get(scope_name,  config["SECRETS"]["DATABRICKS_ACCOUNT_URL"] )
ACCOUNT_API_TOKEN = dbutils.secrets.get(scope_name,  config["SECRETS"]["ACCOUNT_API_TOKEN"] )

HEADERS = {"Authorization": f"Bearer {ACCOUNT_API_TOKEN}", "Content-Type": "application/json"}

In [0]:
def get_group_id(group_name):
    url = f"https://accounts.cloud.databricks.com/api/2.0/accounts/{account_id}/scim/v2/Groups"
    response = requests.get(url, headers=HEADERS, params = {"filter": f"displayName eq {group_name}"} )
    resources = response.json().get("Resources", [])
    # print(resources)
    if resources:
        return resources[0]["id"]
    return None

In [0]:
def create_group(group_name):
    url = f"https://accounts.cloud.databricks.com/api/2.0/accounts/{account_id}/scim/v2/Groups"
    payload = {"displayName": group_name}
    response = requests.post(url, headers=HEADERS, data=json.dumps(payload))
    if response.status_code == 201:
        return response.json()["id"]
    raise Exception(f"Failed to create group: {response.text}")

In [0]:
def add_user_to_group(group_id, user_email):
    # Find user by email
    url = f"https://accounts.cloud.databricks.com/api/2.0/accounts/{account_id}/scim/v2/Users?filter=userName eq \"{user_email}\""
    user_resp = requests.get(url, headers=HEADERS)
    users = user_resp.json().get("Resources", [])
    if not users:
        raise Exception(f"User {user_email} not found in account.")
    user_id = users[0]["id"]


    # Check & Add user to group
    group_url = f"https://accounts.cloud.databricks.com/api/2.0/accounts/{account_id}/scim/v2/Groups/{group_id}"

    group_resp = requests.get(group_url, headers=HEADERS)
    group_members = group_resp.json().get("members", [])

    if any(member.get("value") == user_id for member in group_members):
        print(f" User `{user_email}` already exist.")
        return 

    payload = {
        "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
        "Operations": [{
            "op": "add",
            "path": "members",
            "value": [{"value": user_id}]
        }]
    }
    response = requests.patch(group_url, headers=HEADERS, data=json.dumps(payload))
    print(f"User `{user_email}` added.")
    if response.status_code != 200:
        raise Exception(f"Failed to add user to group: {response.text}")

In [0]:
def grant_permission(table_name, principal, permission):
    query = f"GRANT {permission} ON TABLE {table_name} TO `{principal}`;"
    spark.sql(query)
    print(f"Permission granted to `{principal}`. ")

In [0]:
def execute_creation():
    group_id = get_group_id(group_name)
    if not group_id:
        print(f"Group '{group_name}' not found. Creating...")
        group_id = create_group(group_name)
        print(f"{group_name} created successfully. ")
    else:
        print(f"Group '{group_name}' already exists.")

    # add users to group
    user_list = user_names.split(",")
    for user_email in user_list:
        print(f"Start Adding user '{user_email}' to group '{group_name}' ---> ",end="")
        add_user_to_group(group_id, user_email.strip())
    
    if permissions == '' and table_name=='' and flag == '':
        print("No permission need to grant")
        return

    # Grant permission to table
    if flag.lower() =='group':
        grant_permission(table_name, group_name, permissions)
    else:
        for user_email in user_list:
            grant_permission(table_name, user_email.strip(), permissions)

    # The End #
    print("******The End ********")

In [0]:
execute_creation()